# Dataset Preparation

- input: curated briefsummary
- output: pd.series of ase objects pr pymatgen objects


In [1]:
import sys
sys.path.insert(0, '/home/storage/fortimtb/CuadernoTrabajo/bopfoxfeaturizer/')
import os
import pandas as pd
from BopFoxFeaturizer.Featurizer import Featurizer
from Tools.DatasetTools.Tools import need_to_update
from Tools.DatasetTools.SublatticeSorter import *
from pymatgen.io.ase import AseAtomsAdaptor

# options 

In [2]:
dataset = 'Cr-Co-W'
case='POSCAR-initial' #, 'POSCAR-relaxed']
rescale_by_atoms=True #, False]
subcase = 'rescaled' # ,  'noscaled' ] 
Force= True
CuratedBS = os.path.join(dataset,'CuratedParsedBriefSummary.pkl')

In [3]:
BS = pd.read_pickle(CuratedBS)

In [4]:
Features = Featurizer(BS)

chech that the chemistry resetting is correct!

# Sort Poscar files

In [5]:
searchs = 'POSCAR.initial'

In [6]:
files = get_file_paths(dataset, searchs)

In [7]:
atomsobjectslocation = os.path.join(dataset,'Atomsobjects')
sublatticesortersfile = os.path.join(atomsobjectslocation, 'SORTERS.pkl')
sublatticetagfile = os.path.join(atomsobjectslocation, 'SUBLATICETAGS.pkl')
if need_to_update(sublatticesortersfile) or need_to_update(sublatticetagfile):
    SORTERS, SUBLATICETAGS = get_all_sorters_and_tags(dataset, files)
    SORTERS.to_pickle(sublatticesortersfile)
    SUBLATICETAGS.to_pickle(sublatticetagfile)
else:
    SORTERS = pd.read_pickle(sublatticesortersfile)
    SUBLATICETAGS = pd.read_pickle(sublatticetagfile)

  0%|          | 0/1850 [00:00<?, ?it/s]

# Now I have to pick the atoms objects

In [8]:
#for thiscase, (thisrescale, thissubcase) in product(case, zip(rescale_by_atoms, subcase)):
database = f'{dataset}/**/{case}'
AtomsFile = os.path.join(atomsobjectslocation,f'CrCoW-sorted-{case}-{subcase}-AtomsObjects.pkl')

In [9]:
if not need_to_update(AtomsFile):  #os.path.exists(AtomsFile) and not Force:
    Atoms_Objects = pd.read_pickle(AtomsFile)
else:
    Atoms_Objects, CantMake_Atoms_Object = Features.get_atoms_object(database=database,rescale_by_atoms=True, reset_chemistry=True, file_filter = 'sorted')
    Atoms_Objects.to_pickle(AtomsFile)
    Atoms_Objects.dropna(inplace=True)
pymatgenfile = AtomsFile.replace('AtomsObjects','PymatgenStructures')
Pymatgen_Structures = Atoms_Objects.copy()
if not need_to_update(pymatgenfile):
    Pymatgen_Structures = pd.read_pickle(pymatgenfile)
else:
    Pymatgen_Structures = Atoms_Objects['atoms'].apply(AseAtomsAdaptor.get_structure)
    Pymatgen_Structures['file'] = Atoms_Objects['file']
    Pymatgen_Structures.to_pickle(pymatgenfile)

  0%|          | 0/1701 [00:00<?, ?it/s]

In [10]:
Atoms_Objects.atoms.isna().sum()

0

In [12]:
Atoms_Objects

,atoms,file
Co_pv6W_sv6.C14-BBA.FM,"(Atom('Co', [2.214013537060907, 1.278260741187...",[Cr-Co-W/data/Co_pv-W_sv/POSCAR-initial/C14-BB...
Co_pv6W_sv6.C14-BBA.NM,"(Atom('Co', [2.214013537060907, 1.278260741187...",[Cr-Co-W/data/Co_pv-W_sv/POSCAR-initial/C14-BB...
Cr_pv6W_sv2.D0_19-A3B.FM,"(Atom('Cr', [2.5256473435629325, 2.95106973096...",[Cr-Co-W/data/Cr_pv-W_sv/POSCAR-initial/D0_19-...
Cr_pv6W_sv2.D0_19-A3B.NM,"(Atom('Cr', [2.5256473435629325, 2.95106973096...",[Cr-Co-W/data/Cr_pv-W_sv/POSCAR-initial/D0_19-...
Cr_pv16Co_pv4W_sv10.sigma-CBAAC.FM,"(Atom('Cr', [3.971498114956838, 1.123685211791...",[Cr-Co-W/data/Cr_pv-Co_pv-W_sv/POSCAR-initial/...
...,...,...
Cr_pv10W_sv3.mu-BAAAB.FM,"(Atom('Cr', [0.0, 0.0, 0.0], index=0), Atom('C...",[Cr-Co-W/data/Cr_pv-W_sv/POSCAR-initial/mu-BAA...
Cr_pv20Co_pv2W_sv8.sigma-BAACA.NM,"(Atom('Cr', [3.4303434966356314, 3.43034349663...",[Cr-Co-W/data/Cr_pv-Co_pv-W_sv/POSCAR-initial/...
Cr_pv20Co_pv2W_sv8.sigma-BAACA.FM,"(Atom('Cr', [3.4303434966356314, 3.43034349663...",[Cr-Co-W/data/Cr_pv-Co_pv-W_sv/POSCAR-initial/...
Co_pv13W_sv16.chi-ABAB.NM,"(Atom('Co', [0.0, 0.0, 0.0], index=0), Atom('C...",[Cr-Co-W/data/Co_pv-W_sv/POSCAR-initial/chi-AB...


In [13]:
Atoms_Objects.to_pickle(AtomsFile)

In [14]:
accomodatewrap = Atoms_Objects.atoms.map(lambda a: a.wrap(pretty_translation=True))

In [16]:
BS.index.difference(Atoms_Objects.index)

Index(['Co_pv1.bcc.FM', 'Co_pv1.fcc.FM', 'Co_pv15W_sv38.R-AAAABBBBBBB.NM',
       'Co_pv1W_sv52.R-ABBBBBBBBBB.NM', 'Co_pv21W_sv32.R-AAAAABBBBBB.NM',
       'Co_pv27W_sv26.R-AAAAAABBBBB.NM', 'Co_pv33W_sv20.R-AAAAAAABBBB.NM',
       'Co_pv3W_sv50.R-AABBBBBBBBB.NM', 'Co_pv45W_sv8.R-AAAAAAAAABB.NM',
       'Co_pv47W_sv6.R-AAAAAAAAAAB.NM', 'Co_pv9W_sv44.R-AAABBBBBBBB.NM',
       'Cr_pv1.bcc.FM', 'Cr_pv1.fcc.FM', 'Cr_pv1W_sv52.R-ABBBBBBBBBB.NM',
       'Cr_pv27W_sv26.R-AAAAAABBBBB.NM', 'Cr_pv39W_sv14.R-AAAAAAAABBB.NM',
       'Cr_pv3W_sv50.R-AABBBBBBBBB.NM', 'Cr_pv45W_sv8.R-AAAAAAAAABB.NM',
       'Cr_pv47W_sv6.R-AAAAAAAAAAB.NM'],
      dtype='object')

#  visualization of some structures

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rc('axes.spines', bottom=False, top=False, right=False, left=False)

In [ ]:
from ase.visualize.plot import plot_atoms

In [ ]:
atoms_samples = Atoms_Objects.atoms.sample(n=9)
fig, ax = plt.subplots(3,3, figsize = (20,20))
ax = ax.flatten()
for thisax, (thisname, thisatoms) in  zip(ax, atoms_samples.iteritems()):
    plot_atoms(thisatoms, ax=thisax, rotation = '90x')
#    [spine.set_visible(False) for spine in thisax.spines.values()]
    thisax.set_xticks([])
    thisax.set_yticks([])
    thisax.set_title(thisname)

In [ ]:
somesigmas = Atoms_Objects.atoms[Atoms_Objects.index.str.contains('sigma')].sample(n=9)

In [ ]:
atoms_samples = Atoms_Objects.atoms.sample(n=9)
fig, ax = plt.subplots(3,3, figsize = (20,20))
ax = ax.flatten()
for thisax, (thisname, thisatoms) in  zip(ax, somesigmas.iteritems()):
    plot_atoms(thisatoms, ax=thisax, rotation='90y, 90x, 45y')
    [spine.set_visible(False) for spine in thisax.spines.values()]
    thisax.set_xticks([])
    thisax.set_yticks([])
    thisax.set_title(thisname)

 For the actual visualization of the structures, we should choose one example for each structure and then draw in Vesta or Ovito for good quality figures, including coordination polyhedra etc.

# Curate Dataset to available structures 

There are still some R structures not available in data but present in briefsummaries

In [ ]:
Problems = BS.index.difference(Atoms_Objects.index)

In [ ]:
BS.loc[Problems]

In [ ]:
GoodBS = BS.loc[Atoms_Objects.index]

In [ ]:
GoodBS

In [ ]:
GoodBSFile = os.path.join(dataset, 'FullyCuratedParsedBriefSummary.pkl')

# Prepare Extra features

In [ ]:
Features = Featurizer(GoodBS)

In [ ]:
Features.Mag

In [ ]:
DatasetFeatures =[Features.Mag, Features.StrucNames]

In [ ]:
Features.StrucNames

# Prepare targets 

One target still missing is formation Energy. Some Convenience functions to do this has been set

In [ ]:
import sys
sys.path.insert(0, '/home/storage/fortimtb/CuadernoTrabajo/bopfoxfeaturizer/')
from BopFoxFeaturizer.Featurizer import Featurizer

In [ ]:
ground_states = Features.get_ground_states_energies()

In [ ]:
GoodBS['EF'] = Features.get_formation_energy(ground_states)

In [ ]:
GoodBS['EF']

In [ ]:
GoodBS.to_pickle(GoodBSFile)